First data glance

In [1]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.metrics import root_mean_squared_error
from sklearn.feature_extraction import DictVectorizer
from sklearn.pipeline import make_pipeline

In [ ]:

def prepare_data(
    datapath,
    outcome,
    numerical_features,
    categorical_features,
    pipe=None,
    debug_subset=0.0,
):
    df = pd.read_parquet(datapath)

    if debug_subset:
        frac = float(debug_subset)
        df = df.sample(frac=frac, random_state=42)

    df.tpep_pickup_datetime = pd.to_datetime(df.tpep_pickup_datetime)
    df.tpep_dropoff_datetime = pd.to_datetime(df.tpep_dropoff_datetime)
    df["duration"] = df.tpep_dropoff_datetime - df.tpep_pickup_datetime
    df.duration = df.duration.dt.total_seconds() / 60
    df[categorical_features] = df[categorical_features].astype(str)

    X = df[numerical_features + categorical_features]
    y = df[outcome]

    if pipe is None:
        dv = DictVectorizer()
        pipe = make_pipeline(dv)
        X_train = pipe.fit_transform(X.to_dict(orient="records"))
    else:
        X_train = pipe.transform(X.to_dict(orient="records"))

    return X_train, y, pipe

def train_model(X_train, y, y_test, model):
    model.fit(X_train, y)
    y_pred = model.predict(X_train)
    rmse = root_mean_squared_error(y, y_pred)
    print(f"RMSE: {rmse:.2f}")

: 

In [3]:
NUMERICAL_FEATURES = ["trip_distance", "passenger_count", "tip_amount", "total_amount", "tolls_amount"]
CATEGORICAL_FEATURES = ['PULocationID', 'DOLocationID', "payment_type"]
OUTCOME = 'duration'
PIPE = None

train_path = '/workspaces/MLOps-zoomcamp/data/yellow_tripdata_2020-01.parquet'
test_path = '/workspaces/MLOps-zoomcamp/data/yellow_tripdata_2020-02.parquet'

X_train, y_train, pipe = prepare_data(train_path, OUTCOME, NUMERICAL_FEATURES, CATEGORICAL_FEATURES, pipe=None, debug_subset=0.1)
X_test, y_test, _ = prepare_data(test_path, OUTCOME, NUMERICAL_FEATURES, CATEGORICAL_FEATURES, pipe, debug_subset=0.1)
model = LinearRegression()

train_model(X_train, y_train, y_test, model)

: 

: 